# DATA 37100 — Final Project Analysis
**Student:** Ben Bentley  
**Date:** March 2026

## Research Question
**How do diffusion timesteps (T) and prediction target (eps vs x0) affect sample sharpness and failure modes on MNIST?**

## Model Coverage
- **Baseline 1:** Diffusion (MNIST)
- **Baseline 2:** DCGAN (MNIST)
- **Controlled Experiment:** Diffusion two-knob grid (T × target)

## 1. Setup

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Set paths relative to repo root
REPO_ROOT = Path('.').resolve()
DIFF_OUT = REPO_ROOT / 'untrack/outputs/final/diffusion'
GAN_OUT = REPO_ROOT / 'untrack/outputs/final/gan'

print(f"Diffusion outputs: {DIFF_OUT}")
print(f"GAN outputs: {GAN_OUT}")

Diffusion outputs: /Users/benbentley/Documents/School/UChicago/Winter 2026/Intro to AI Deep Learning and Generative AI/BenBentley_DATA37100_Final/final/draft/untrack/outputs/final/diffusion
GAN outputs: /Users/benbentley/Documents/School/UChicago/Winter 2026/Intro to AI Deep Learning and Generative AI/BenBentley_DATA37100_Final/final/draft/untrack/outputs/final/gan


## 2. Baseline Results

### 2.1 Diffusion Baseline (T=200, eps target)

In [2]:
# Load diffusion baseline summary
diff_baseline_dirs = sorted([d for d in DIFF_OUT.glob('*') if d.is_dir() and 'T200' in d.name])
if diff_baseline_dirs:
    diff_baseline = diff_baseline_dirs[0]
    diff_summary = json.loads((diff_baseline / 'summary.json').read_text())
    diff_args = json.loads((diff_baseline / 'run_args.json').read_text())
    
    print("Diffusion Baseline Configuration:")
    print(f"  T: {diff_args.get('T')}")
    print(f"  target: {diff_args.get('target')}")
    print(f"  base_ch: {diff_args.get('base_ch')}")
    print(f"  Runtime: {diff_summary.get('seconds', 0):.1f}s")
    print(f"  Device: {diff_summary.get('device')}")
else:
    print("Run baselines first: bash final/draft/run_baselines.sh")

Run baselines first: bash final/draft/run_baselines.sh


In [3]:
# Display diffusion baseline samples
if diff_baseline_dirs:
    sample_imgs = sorted(diff_baseline.glob('*.png'))
    if sample_imgs:
        img = mpimg.imread(str(sample_imgs[0]))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.title(f"Diffusion Baseline Samples (T={diff_args.get('T')}, {diff_args.get('target')})")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

### 2.2 DCGAN Baseline

In [4]:
# Load GAN baseline summary
gan_baseline_dirs = sorted([d for d in GAN_OUT.glob('*') if d.is_dir()])
if gan_baseline_dirs:
    gan_baseline = gan_baseline_dirs[0]
    gan_summary = json.loads((gan_baseline / 'summary.json').read_text())
    gan_args = json.loads((gan_baseline / 'run_args.json').read_text())
    
    print("DCGAN Baseline Configuration:")
    print(f"  max_steps: {gan_args.get('max_steps')}")
    print(f"  lr: {gan_args.get('lr')}")
    print(f"  base_ch: {gan_args.get('base_ch')}")
    print(f"  z_dim: {gan_args.get('z_dim')}")
    print(f"  Runtime: {gan_summary.get('seconds', 0):.1f}s")
    print(f"  Device: {gan_summary.get('device')}")

In [5]:
# Display GAN baseline samples
if gan_baseline_dirs:
    sample_imgs = sorted(gan_baseline.glob('*.png'))
    if sample_imgs:
        img = mpimg.imread(str(sample_imgs[0]))
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.title(f"DCGAN Baseline Samples ({gan_args.get('max_steps')} steps)")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## 3. Controlled Experiment: T × target Grid

**Experimental Design:**
- **Knob 1:** T (diffusion timesteps) ∈ {100, 200, 400}
- **Knob 2:** target (prediction parameterization) ∈ {eps, x0}
- **Total runs:** 6
- **Control variables:** base_ch=64, seed=42, dataset=MNIST, epochs=1

In [6]:
# Load experiment results
results_csv = DIFF_OUT / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(f"Loaded {len(df)} experiment runs")
    display(df[['run_dir']].head(10))
else:
    print("Run experiment first: bash final/draft/run_experiment.sh")
    df = None

Run experiment first: bash final/draft/run_experiment.sh


In [7]:
# Extract run parameters and metrics
if df is not None:
    def read_json(p: Path):
        return json.loads(p.read_text(encoding='utf-8'))

    rows = []
    for rd in df['run_dir']:
        rd = Path(rd)
        args = read_json(rd / 'run_args.json') if (rd / 'run_args.json').exists() else {}
        summ = read_json(rd / 'summary.json') if (rd / 'summary.json').exists() else {}
        
        row = {
            'run_name': rd.name,
            'T': args.get('T'),
            'target': args.get('target'),
            'base_ch': args.get('base_ch'),
            'runtime_sec': summ.get('seconds', 0),
            'device': summ.get('device'),
        }
        rows.append(row)

    summary_df = pd.DataFrame(rows)
    print("\nExperiment Summary:")
    display(summary_df)
    
    # Runtime statistics
    print(f"\nAverage runtime: {summary_df['runtime_sec'].mean():.1f}s")
    print(f"Total runtime: {summary_df['runtime_sec'].sum():.1f}s")

### 3.1 Visual Comparison Grid

In [8]:
# Create side-by-side comparison of all experiment runs
if df is not None:
    def find_pngs(run_dir: Path):
        candidates = []
        if (run_dir / 'samples').exists():
            candidates += sorted((run_dir / 'samples').glob('*.png'))
        candidates += sorted(run_dir.glob('*.png'))
        # Prefer files with 'grid' or 'sample' in name
        candidates = sorted(candidates, key=lambda p: (
            ('grid' not in p.name.lower() and 'sample' not in p.name.lower()),
            p.stat().st_mtime
        ))
        return candidates

    imgs = []
    titles = []
    for _, row in summary_df.iterrows():
        rd = DIFF_OUT / row['run_name']
        pngs = find_pngs(rd)
        if pngs:
            imgs.append(pngs[0])
            titles.append(f"T={row['T']}, {row['target']}")

    n = len(imgs)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    
    plt.figure(figsize=(5*ncols, 5*nrows))
    for i, (p, t) in enumerate(zip(imgs, titles), start=1):
        ax = plt.subplot(nrows, ncols, i)
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(t, fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### 3.2 Effect of T (Timesteps)

**Hypothesis:** More timesteps should produce sharper samples by allowing finer-grained noise removal.

**Observations:**
- *[Fill in after running experiments: describe visual differences between T=100, 200, 400]*
- *[Note any artifacts or quality degradation at extreme values]*

### 3.3 Effect of target (eps vs x0)

**Background:** 
- `eps` predicts noise to remove
- `x0` predicts the clean image directly

**Hypothesis:** `x0` may produce sharper early samples but struggle with stability; `eps` may be more stable but slower to sharpen.

**Observations:**
- *[Fill in after running experiments: compare sharpness, artifacts, failure modes]*

### 3.4 Interaction Effects

**Question:** Does the effect of T depend on the target parameterization?

**Observations:**
- *[Describe any non-linear interactions between T and target]*

## 4. Failure Modes (Required)

### 4.1 Mode Collapse (Expected in GAN)

**Evidence:**
- *[Check DCGAN samples for diversity: do we see all 10 digits?]*

**Likely Cause:**
- Adversarial training instability — generator may exploit discriminator weaknesses

---

### 4.2 Blurry Samples at Low T (Expected in Diffusion)

**Evidence:**
- *[Compare T=100 vs T=400 samples for sharpness]*

**Likely Cause:**
- Insufficient denoising steps — discrete approximation of continuous SDE is too coarse

---

### 4.3 Other Artifacts

**Observed:**
- *[Note any checkerboard patterns, color shifts, or structural failures]*

**Hypothesis:**
- *[Mechanistic explanation based on model architecture or training dynamics]*

## 5. Conclusions

### Key Findings
1. *[Main result about T's effect on sample quality]*
2. *[Main result about target parameterization]*
3. *[Comparison: diffusion vs GAN sample characteristics]*

### One Surprising Result
- *[Something unexpected from the experiment]*

### One Limitation
- *[What this study doesn't capture — e.g., only 1 epoch, only MNIST, etc.]*

### One Next Step
- *[What experiment would you run next to extend this work?]*

---

**Total experiment time:** ~[X] minutes  
**Hardware:** [CPU/GPU type]  
**Analysis time:** ~[Y] hours